In [2]:
import pandas as pd
import numpy as np

def compute_aqi(df):
    """
    Compute AQI using weighted pollutants and scale to realistic range.
    This gives AQI values roughly comparable to official AQI scale (0-500).
    """
    # Weighted sum of pollutants
    df['AQI_raw'] = (
        0.35 * df['PM2.5'] +
        0.25 * df['PM10'] +
        0.20 * df['NO2'] +
        0.20 * df['O3']
    )
    
    # Scale raw AQI to approximate 0-300 range
    # Adjust factor if needed based on data distribution
    df['AQI'] = df['AQI_raw'] * 10
    
    # Clip extreme outliers for stability
    df['AQI'] = df['AQI'].clip(lower=df['AQI'].quantile(0.01),
                                upper=df['AQI'].quantile(0.99))
    
    return df

def create_features(df):
    """
    Create time-series lag and rolling features for LSTM input.
    """
    df['lag1'] = df['AQI'].shift(1)
    df['lag2'] = df['AQI'].shift(2)
    df['rolling_mean_3'] = df['AQI'].rolling(3).mean()
    df['rolling_std_3'] = df['AQI'].rolling(3).std()
    
    df.dropna(inplace=True)
    return df

if __name__ == "__main__":
    # Load dataset
    df = pd.read_csv("data/ancona_data.csv")
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Compute AQI and features
    df = compute_aqi(df)
    df = create_features(df)
    
    # Display sample
    print(df[['Date','AQI','lag1','lag2','rolling_mean_3','rolling_std_3']].head())


FileNotFoundError: [Errno 2] No such file or directory: 'data/ancona_data.csv'

In [1]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.feature_engineering import compute_aqi, create_features

print("Loading dataset...")
df = pd.read_csv("data/ancona_data.csv")
df['Date'] = pd.to_datetime(df['Date'])

print("Engineering features...")
df = compute_aqi(df)
df = create_features(df)

X = df[['lag1','lag2','rolling_mean_3','rolling_std_3']]
y = df['AQI']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, shuffle=False)

print("Training Random Forest model...")
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Evaluating model...")
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print("\nModel Performance:")
print("MAE  :", round(mae,3))
print("RMSE :", round(rmse,3))
print("R2   :", round(r2,4))

joblib.dump(model, "models/rf_model.pkl")
print("\nModel saved successfully in models/rf_model.pkl")


ModuleNotFoundError: No module named 'src'